# Convert NAT-Files (HRV Band) to GeoTiff as multi-threaded Process
## Warp and Clip to Region Western Austria
Conda-Env: Scikit-Learn Requirements: Windows 7-zip Installation (64bit)

In [ ]:
# import necessary python packages
### import py7zr
import os
import gzip
import subprocess
import shutil
import numpy as np
import gdal
import osr
import pyresample as pr
from satpy import Scene
import datetime
from datetime import timedelta
from multiprocessing.dummy import Pool as ThreadPool
import time
import pandas as pd

In [ ]:
def wrapper_nat2geotiff(eumetsat_native_output_path, eumetsat_geotiff_timestamped_path, eumetsat_archive_path, filename):
    print("# Process File:  " + filename)
    if (filename.endswith(".gz")) or filename.endswith(".7z") or filename.endswith(".zip") or filename.endswith(".bz2"):
        if (filename.endswith(".gz")) or filename.endswith(".7z"):
            plain_filename= os.path.basename(filename)[:-7]
        else:
            plain_filename= os.path.basename(filename)[:-8]
        rounded_timestamp_filename = round_filename_to_quarter(filename = os.path.basename(filename)[0:74])
        areas = ["Vorarlberg","Lake of Constance"]
        for area in areas:
            area_id = "Austria West"
            description = "Geographical Coordinate System clipped on Western Austria Region"
            proj_id = "Austria"
            proj_dict = {"proj": "longlat", "ellps": "WGS84", "datum": "WGS84"}
            if area == "Vorarlberg":
                llx = 9
                lly = 46
                urx = 11
                ury = 48
                extend='Clip_Vorarlberg'
            else:
                llx = 7
                lly = 45
                urx = 13
                ury = 50
                extend='Clip_Lake_of_Constance'
            resolution = 0.01
            width = int((urx - llx) / resolution)
            height = int((ury - lly) / resolution)
            area_extent = (llx,lly,urx,ury)
            area_def = pr.geometry.AreaDefinition(area_id, proj_id, description, proj_dict, width, height, area_extent)
            datasets = ['IR_VIS_WV','HRV']
            for dataset in datasets:
                archive_filename = os.path.join(eumetsat_archive_path, filename)
                geotiff_filename = os.path.join(eumetsat_geotiff_timestamped_path, extend, rounded_timestamp_filename + "_{}.tif".format(dataset))
                native_filename = os.path.join(eumetsat_native_output_path, plain_filename + ".nat")
                if not os.path.isfile(geotiff_filename):
                    if not os.path.isfile(native_filename):
                            unzip_command = ['C:\\Program Files\\7-Zip\\7z.exe', 'e', "-o"+ eumetsat_native_output_path, archive_filename, '-y']
                            subprocess.call(unzip_command)
                    reader = "seviri_l1b_native"
                    nat2tif(file = native_filename, calibration = "radiance", area_def = area_def, dataset = dataset,
                            reader = reader, outdir = eumetsat_geotiff_timestamped_path, label = dataset,
                        dtype = "float32", radius = 16000, epsilon = 0.5, nodata = -3.4E+38, outfile = geotiff_filename)
        if os.path.exists(native_filename):
            try:
                os.remove(native_filename)
            except:
                shutil.copyfile(native_filename, os.path.join(eumetsat_archive_path + "\\..\\defekt", plain_filename + ".nat"))
                os.remove(native_filename)
                pass

In [ ]:
def nat2tif(file, calibration, area_def, dataset, reader, outdir, label, dtype, radius, epsilon, nodata, outfile):
  scn = Scene(filenames = {reader: [file]})
  if dataset == 'HRV':
    bands = ['HRV']
  else:
    bands = ['VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
  bandnr = 1
  for band in bands:
    scn_names = scn.all_dataset_names()
    if band not in scn_names:
      raise Exception("Specified dataset is not available.")
    scn.load([band], calibration=calibration)
    lons, lats = scn[band].area.get_lonlats()
    swath_def = pr.geometry.SwathDefinition(lons=lons, lats=lats)
    values = scn[band].values
    lons = lons.astype(dtype)
    lats = lats.astype(dtype)
    values = values.astype(dtype)
    values = pr.kd_tree.resample_nearest(swath_def, values, area_def,
                                              radius_of_influence=radius, epsilon=epsilon, fill_value=False)
    if not os.path.exists(outdir):
      os.makedir(outdir)
    cols = values.shape[1]
    rows = values.shape[0]
    pixelWidth = (area_def.area_extent[2] - area_def.area_extent[0]) / cols
    pixelHeight = (area_def.area_extent[1] - area_def.area_extent[3]) / rows
    originX = area_def.area_extent[0]
    originY = area_def.area_extent[3]
    if bandnr == 1:
        dst_datatype = gdal.GDT_Float32
        driver = gdal.GetDriverByName("GTiff")
        outRaster = driver.Create(outfile, cols, rows, len(bands), dst_datatype, [ 'COMPRESS=ZSTD', 'PREDICTOR=3', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ] )
        outRaster.SetGeoTransform((originX, pixelWidth, 0, originY, 0, pixelHeight))
    outRaster.GetRasterBand(bandnr).SetDescription(band)
    outband = outRaster.GetRasterBand(bandnr)
    outband.WriteArray(np.array(values))
    outband.SetNoDataValue(nodata)
    outRasterSRS = osr.SpatialReference()
    outRasterSRS.ImportFromEPSG(4326)
    outRaster.SetProjection(outRasterSRS.ExportToWkt())
    bandnr = bandnr + 1
  outband = None
  outRaster.FlushCache()
  outRaster = None
  del file, scn, outband, outRaster

In [ ]:
def round_filename_to_quarter(filename):
    date_object = datetime.datetime.strptime(filename[24:38], '%Y%m%d%H%M%S')
    rounded = date_object - (date_object - date_object.min) % timedelta(minutes=15)
    rounded_str=rounded.strftime("%Y-%m-%d %H_%M_%S")
    return rounded_str

In [ ]:
if __name__ == '__main__':
    from concurrent.futures import ThreadPoolExecutor
    from concurrent.futures import as_completed
    from itertools import repeat
    eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
    eumetsat_native_output_path = eumetsat_path + "\\Native"
    eumetsat_geotiff_timestamped_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\TEST"
    eumetsat_archive_path = "F:\\EUMETSAT\\TEST"
    if not os.path.exists(eumetsat_native_output_path):
         os.makedirs(eumetsat_native_output_path)
    if not os.path.exists(eumetsat_geotiff_timestamped_path):
         os.makedirs(eumetsat_geotiff_timestamped_path)
    files = []
    for file in os.listdir(eumetsat_archive_path):
        files.append(file)
    with ThreadPoolExecutor(max_workers = 8) as executor:
        results = executor.map(wrapper_nat2geotiff, repeat(eumetsat_native_output_path), repeat(eumetsat_geotiff_timestamped_path), repeat(eumetsat_archive_path), files)
    for result in results:
        print(result)